# 01. D1: 반복 요청 재생 (C0/C1/C2/C3, 운영 설정)

요청 4,000개 trace를 반복 비율 0/10/30/50% × seed 3개로 만든다. 같은 (비율, seed)의 trace는 네 비교군이 똑같이 받는다. 각 비교군은 빈 캐시에서 시작하고, 미스면 운영 1.0 라우터가 고른 단계의 **기록 답변·비용**을 쓰고 캐시에 넣는다. 정답 여부로 저장을 거르지 않는다.

- C0 캐시 없음 · C1 exact만 · C2 exact → 코사인 0.80 · C3 exact → 코사인 → 리랭커(현재 운영 구성)
- 다른 질문의 답이 나가면 재사용 관계 오류(F)로 세고, 그 답을 **현재 질문 기준으로 다시 채점**한다. 채점할 수 없으면 unknown으로 두고 품질을 보수·낙관 범위로 낸다.
- 보조: 용량 제한(LRU, `updated_at` 기준) 민감도. 운영에는 용량 제한이 없다.
- exact 히트가 다른 질문의 답을 내면 정규화 버그로 보고 즉시 멈춘다.

리랭커 점수는 Colab GPU에서 계산한다. D2와 무관하게 먼저 실행할 수 있다.

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

RUN_ID = os.environ.get("CACHE_EXP_RUN_ID", "run001")  # 새 계획·재시작이면 새 ID
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    LLM_ROUTER = Path("/content/drive/MyDrive/LLM_Router")
    KIT = LLM_ROUTER / "cache-experiment"
    zip_path = KIT / "source/cache-source.zip"
    expected = (KIT / "source/cache-source.sha256").read_text().split()[0]
    assert hashlib.sha256(zip_path.read_bytes()).hexdigest() == expected, "source ZIP SHA mismatch"
    REPO = Path("/content/cache-source")
    with zipfile.ZipFile(zip_path) as z:
        for name in z.namelist():
            assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
        z.extractall(REPO)
    for name, digest in json.loads((REPO / "source-bundle-manifest.json").read_text()).items():
        assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == digest, name
    HERE = REPO / "ml/notebooks/cache_experiment"
    DATA_ROOT, OUTPUT_ROOT = KIT / "data-root", KIT / "runs"
    ROUTER_DIR = LLM_ROUTER / "transfer-experiments/inputs/experiment-A/models/r0"
    os.environ.setdefault("CACHE_EXP_DRIVE_LOGGER", str(LLM_ROUTER / "retraining-effect/logger/logger-log.parquet"))
    assert sys.version_info[:2] in {(3, 11), (3, 12)}, "런타임 버전 2026.07(Python 3.12)을 선택하세요"
    if not (REPO / ".installed").exists():
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.5.1",
                        "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(HERE / "requirements-colab.txt")], check=True)
        (REPO / ".installed").write_text("ok")
        # Colab already imported its own numpy; mixing it with the pinned one breaks
        # transformers. Restart the runtime (files in /content survive) and run all again.
        print("설치 완료. 런타임을 재시작합니다. 재시작되면 '모두 실행'을 다시 누르세요.")
        os.kill(os.getpid(), 9)
    import numpy
    pinned = next(l.split("==")[1].strip() for l in (HERE / "requirements-colab.txt").read_text().splitlines()
                  if l.startswith("numpy=="))
    assert numpy.__version__ == pinned, (
        f"numpy {numpy.__version__} != {pinned}: 런타임 → 세션 다시 시작 후 '모두 실행'을 다시 누르세요")
else:
    # 로컬: 저장소 루트의 .venv-cache 커널. 입력은 ml/data·ml/artifacts가 있는 체크아웃,
    # 산출물은 Drive 동기화 폴더의 kit runs/ 가 있으면 그곳(Colab과 같은 run을 이어 쓴다).
    HERE = Path.cwd()
    DATA_ROOT = Path(os.environ.get("CACHE_EXP_DATA_ROOT", HERE.parents[2]))
    synced = sorted(Path.home().glob("Library/CloudStorage/GoogleDrive-*/*/LLM_Router/cache-experiment/runs"))
    OUTPUT_ROOT = os.environ.get("CACHE_EXP_OUTPUT_ROOT") or (synced[0] if synced else None)
    ROUTER_DIR = None  # config router.package_path
    found = sorted(Path.home().glob("Library/CloudStorage/GoogleDrive-*/*/LLM_Router/retraining-effect/logger/logger-log.parquet"))
    if found:
        os.environ.setdefault("CACHE_EXP_DRIVE_LOGGER", str(found[0]))
CONFIG = Path(os.environ.get("CACHE_EXP_CONFIG", HERE / "experiment_config.json"))
sys.path.insert(0, str(HERE))

import pandas as pd
import cache_experiment as ce
import models
import pipeline

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
run = pipeline.Run(DATA_ROOT, RUN_ID, CONFIG, OUTPUT_ROOT)
print("run directory:", run.root)

In [ ]:
runtime = models.runtime_versions(run.config)
reranker = models.Reranker(run.config, device=run.config["inference_device"])

## 1. trace 봉인

In [ ]:
traces = pipeline.make_traces(run)
traces[["trace_id", "repeat_fraction", "seed", "repeats", "unique_questions", "requests_by_source"]]

## 2. 재생

이미 만든 비교군 로그는 다시 계산하지 않고 읽는다(중단 후 재실행 가능).

In [ ]:
summary = pipeline.run_d1(run, reranker)
main = summary[summary.arm.isin(ce.ARMS)]
main.pivot_table(index="repeat_fraction", columns="arm",
                 values=["hit_rate", "call_reduction", "false_reuse_among_hits", "quality_change_conservative"],
                 aggfunc="mean").round(4)

## 3. 질문 그룹 paired bootstrap (C0 대비)

In [ ]:
boot = pipeline.d1_bootstrap(run)
boot[boot.metric.isin(["call_reduction", "false_reuse_among_hits", "quality_change_conservative"])].round(4)

## 4. 용량 제한 민감도

In [ ]:
import report
report.capacity_table(summary).round(4)